In [95]:
import random
import sys
from linecache import cache
from multiprocessing.spawn import prepare

import pandas as pd
from rsa.prime import are_relatively_prime
import importlib

import prepare_data
importlib.reload(prepare_data)
from prepare_data import *


sys.path.append('../result_analysis')
import CARE_scores as care
importlib.reload(care)


<module 'CARE_scores' from 'C:\\Users\\olab0\\OneDrive\\Pulpit\\Pulpit_\\Studies\\Informatyka\\ProjektGrupowy\\repo\\Anomaly-Detection-for-Wind-Turbines\\anomaly_detection\\../result_analysis\\CARE_scores.py'>

In [96]:
dataset_dir_path = '../../../data/Care_To_Compare/Wind Farm A/Wind Farm A/datasets'
event_file_path = '../../../data/Care_To_Compare/Wind Farm A/Wind Farm A/event_info.csv'


In [44]:
class AllNormal():
    def AllNormal(self):
        pass
    
    def fit(self,x_train,y_train):
        pass
    
    def predict(self, x_test):
        return [False for i in range(x_test.shape[0])]

In [43]:
class AllAnomaly():
    def AllAnomaly(self):
        pass
    
    def fit(self,x_train,y_train):
        pass
    
    def predict(self, x_test):
        return [True for i in range(x_test.shape[0])]

In [45]:
class AllRandom():
    def AllNormal(self):
        pass
    
    def fit(self,x_train,y_train):
        pass
    
    def predict(self, x_test):
        return [random.choice([True, False]) for i in range(x_test.shape[0])]

In [97]:
data = pd.read_csv(event_file_path, sep = ';')
rows_used = [4, 0, 12, 15]
train, prediction = generate_train_prediction(rows_used, data)
prediction = generate_ground_truth(data, prediction)

In [98]:
train[0]

,time_stamp,asset_id,id,train_test,status_type_id,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,...,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg
0,2022-07-28 13:20:00,11,0,train,0,31.0,152.0,48.7,3.9,3.9,...,-2090.0,0.0,0.0,-1185.0,-2090.0,0.4,2.6,0.0,0.8,34.0
1,2022-07-28 13:30:00,11,1,train,0,31.0,86.1,150.9,6.0,6.0,...,-1627.0,0.0,0.0,-1050.0,-1627.0,0.0,0.0,0.0,0.0,34.0
2,2022-07-28 13:40:00,11,2,train,0,31.0,115.2,69.6,6.3,6.3,...,-1624.0,0.0,0.0,-1043.0,-1624.0,0.0,0.0,0.0,0.0,34.0
3,2022-07-28 13:50:00,11,3,train,0,32.0,129.3,-29.1,6.0,5.9,...,-212.0,-9540.0,0.0,40124.0,-9753.0,9.5,14.0,0.0,4.8,34.0
4,2022-07-28 14:00:00,11,4,train,0,32.0,137.7,26.4,7.1,6.9,...,0.0,-25215.0,0.0,99360.0,-25215.0,13.1,14.9,10.8,1.3,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52058,2023-07-28 12:30:00,11,52058,train,0,31.0,224.8,31.9,2.3,2.3,...,-689.0,0.0,0.0,-677.0,-689.0,1.2,2.0,0.0,0.8,35.0
52059,2023-07-28 12:40:00,11,52059,train,0,31.0,199.5,6.6,2.1,2.1,...,-599.0,0.0,0.0,-621.0,-599.0,0.2,1.6,0.0,0.6,35.0
52060,2023-07-28 12:50:00,11,52060,train,0,32.0,113.2,-79.7,2.0,2.0,...,-584.0,0.0,0.0,-617.0,-584.0,0.1,1.5,0.0,0.3,36.0
52061,2023-07-28 13:00:00,11,52061,train,0,32.0,246.2,53.3,2.1,2.1,...,-558.0,0.0,0.0,-587.0,-558.0,0.2,1.6,0.0,0.5,36.0


In [50]:
rows_anomalous = [4, 0]
rows_normal = [12, 15]

## Training and predicting

In [99]:
AN = AllNormal()
AA = AllAnomaly()
AR = AllRandom()

an_y_pred, aa_y_pred, ar_y_pred = {},{},{}

for key in rows_used:
    train_set = train[key]
    prediction_set = prediction[key]
    X_train = train_set
    y_train = [False for i in range(X_train.shape[0])]
    X_test = prediction_set.drop(['is_anomaly'], axis = 1)
    y_true = prediction_set['is_anomaly']
    
    AN.fit(X_train, y_train)
    y_pred = AN.predict(X_test)
    an_y_pred.update({key:y_pred})
    
    AA.fit(X_train, y_train)
    y_pred = AA.predict(X_test)
    aa_y_pred.update({key:y_pred})
    
    AR.fit(X_train, y_train)
    y_pred = AR.predict(X_test)
    ar_y_pred.update({key:y_pred})
    

## Results


### CARE coverage 
Should be calculated only for anomalous datasets

In [100]:
aa_cov, an_cov, ar_cov = [], [], []

for key in rows_anomalous:
    y_true = prediction[key]['is_anomaly']
    
    y_pred = an_y_pred[key]
    cov = care.CARE_coverage(y_true, y_pred)
    an_cov.append(cov)
    
    y_pred = aa_y_pred[key]
    cov = care.CARE_coverage(y_true, y_pred)
    aa_cov.append(cov)
    
    y_pred = ar_y_pred[key]
    cov = care.CARE_coverage(y_true, y_pred)
    ar_cov.append(cov)
    
print("Average coverage for AllNormal: " + str(sum(an_cov) / len(an_cov)))
print("Average coverage for AllAnomaly: " + str(sum(aa_cov) / len(aa_cov)))
print("Average coverage for AllRandom: " + str(sum(ar_cov) / len(ar_cov)))
    

Average coverage for AllNormal: 0.0
Average coverage for AllAnomaly: 0.8261788488584528
Average coverage for AllRandom: 0.712702009098403


C:\Users\olab0\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\olab0\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### CARE accuracy 
Should be calculated only for normal datasets

In [101]:
aa_acc, an_acc, ar_acc = [], [], []

for key in rows_normal:
    y_true = prediction[key]['is_anomaly']
    
    y_pred = an_y_pred[key]
    acc = care.CARE_accuracy(y_true, y_pred)
    an_acc.append(acc)
    
    y_pred = aa_y_pred[key]
    acc = care.CARE_accuracy(y_true, y_pred)
    aa_acc.append(acc)
    
    y_pred = ar_y_pred[key]
    acc = care.CARE_accuracy(y_true, y_pred)
    ar_acc.append(acc)
    
print("Average accuracy for AllNormal: " + str(sum(an_acc) / len(an_acc)))
print("Average accuracy for AllAnomaly: " + str(sum(aa_acc) / len(aa_acc)))
print("Average accuracy for AllRandom: " + str(sum(ar_acc) / len(ar_acc)))

Average accuracy for AllNormal: 1.0
Average accuracy for AllAnomaly: 0.0
Average accuracy for AllRandom: 0.4966255302738141


### CARE Earliness
Should be calculated only for anomalous datasets

In [102]:
aa_ear, an_ear, ar_ear = [], [], []

for key in rows_anomalous:
    y_true = prediction[key]['is_anomaly']
    
    y_pred = an_y_pred[key]
    ear = care.CARE_earliness(y_true, y_pred)
    an_ear.append(ear)
    
    y_pred = aa_y_pred[key]
    ear = care.CARE_earliness(y_true, y_pred)
    aa_ear.append(ear)
    
    y_pred = ar_y_pred[key]
    ear = care.CARE_earliness(y_true, y_pred)
    ar_ear.append(ear)
    
print("Average Earliness for AllNormal: " + str(sum(an_ear) / len(an_ear)))
print("Average Earliness for AllAnomaly: " + str(sum(aa_ear) / len(aa_ear)))
print("Average Earliness for AllRandom: " + str(sum(ar_ear) / len(ar_ear)))

Average Earliness for AllNormal: 0.0
Average Earliness for AllAnomaly: 1.0
Average Earliness for AllRandom: 0.5072921861866918


### CARE Reliability
Calculated by dataset (event), not by timestamp

In [103]:
event_label_true = []
event_label_pred_aa = []
event_label_pred_ar = []
event_label_pred_an = []

for key in rows_used:
    if key in rows_anomalous:
        event_label_true.append(True)
    else:
        event_label_true.append(False)
    
    status_id = prediction[key]['status_type_id']
    
    y_pred = an_y_pred[key]
    event_label_pred_an.append(care.compute_event_label(y_pred, status_id, threshold=72))
    
    y_pred = aa_y_pred[key]
    event_label_pred_aa.append(care.compute_event_label(y_pred, status_id, threshold=72))
    
    y_pred = ar_y_pred[key]
    event_label_pred_ar.append(care.compute_event_label(y_pred, status_id, threshold=72))

aa_rel = care.CARE_reliability(event_label_true, event_label_pred_aa)
an_rel = care.CARE_reliability(event_label_true, event_label_pred_an)
ar_rel = care.CARE_reliability(event_label_true, event_label_pred_ar)
    
print("Reliability for AllNormal: " + str(an_rel))
print("Reliability for AllAnomaly: " + str(aa_rel))
print("Reliability for AllRandom: " + str(ar_rel))
        

    


Reliability for AllNormal: 0
Reliability for AllAnomaly: 0.3571428571428571
Reliability for AllRandom: 0


C:\Users\olab0\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\olab0\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
